In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

This is the python code used to export weights and bias of the model

In [ ]:
df = pd.read_csv('motor_control_dataset_normalized.csv')

X = df[['Speed_ref', 'Speed', 'Current', 'Bus_voltage']].values
y = df[['PWM']].values

#convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

#create dataloader for batching
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

#define the 4-8-1 MLP Architecture
class MotorControlMLP(nn.Module):
    def __init__(self):
        super(MotorControlMLP, self).__init__()
        #input layer (4) to Hidden layer (8)
        self.fc1 = nn.Linear(4, 8)
        #ReLU
        self.relu = nn.ReLU()
        #hidden layer (8) to Output layer (1)
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MotorControlMLP()

#define Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

#training Loop
epochs = 100
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.6f}")

print("\nTraining Complete!")

#extract weight and bias 
print("\n--- EXTRACTED WEIGHTS & BIASES ---")
weights_fc1 = model.fc1.weight.data.numpy()
bias_fc1 = model.fc1.bias.data.numpy()
weights_fc2 = model.fc2.weight.data.numpy()
bias_fc2 = model.fc2.bias.data.numpy()

print(f"\nLayer 1 Weights (4 inputs x 8 neurons): \n{weights_fc1}")
print(f"Layer 1 Biases (8 neurons): \n{bias_fc1}")
print(f"\nLayer 2 Weights (8 inputs x 1 output): \n{weights_fc2}")
print(f"Layer 2 Bias (1 output): \n{bias_fc2}")